In [1]:
import csv
import hashlib
import json
import os
import random
import re
import socket
import time
import xml.etree.ElementTree as ET
from collections import defaultdict
from datetime import date, timedelta
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

EMAIL = os.environ.get('NCBI_EMAIL', '')
API_KEY = os.environ.get('NCBI_API_KEY', '')
DATE_FROM = date(2012, 1, 1)
DATE_TO = date(2026, 10, 10)
FIRST_REVIEW_BATCH = 120
OUTPUT = Path.cwd() / 'VirusIdentity_Search_2012_2026_10_10'
CACHE = OUTPUT / 'ncbi_cache'
CACHE.mkdir(parents=True, exist_ok=True)
USE_CACHE = True
RETRIES = 7
TIMEOUT_SECONDS = 40
EFETCH_BATCH = 80
ESEARCH_PAGE = 250
NCBI = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/'

VIRUS = '''(
    virus[tiab] OR viruses[tiab] OR viral[tiab] OR "viral infection"[tiab]
    OR "respiratory infection"[tiab] OR "viral pneumonia"[tiab]
    OR influenza[tiab] OR COVID-19[tiab] OR SARS-CoV-2[tiab]
    OR dengue[tiab] OR chikungunya[tiab] OR zika[tiab]
    OR "respiratory syncytial"[tiab] OR RSV[tiab] OR rhinovirus[tiab]
    OR adenovirus[tiab] OR parainfluenza[tiab] OR metapneumovirus[tiab]
    OR coronavirus[tiab] OR Ebola[tiab] OR enterovirus[tiab]
    OR norovirus[tiab] OR rotavirus[tiab] OR herpesvirus[tiab]
    OR HIV[tiab] OR "hepatitis B"[tiab] OR "hepatitis C"[tiab]
)'''

METHOD = '''(
    "machine learning"[tiab] OR "deep learning"[tiab]
    OR "artificial intelligence"[tiab] OR "supervised learning"[tiab]
    OR "random forest"[tiab] OR "random forests"[tiab]
    OR "support vector"[tiab] OR SVM[tiab] OR XGBoost[tiab]
    OR "gradient boosting"[tiab] OR CatBoost[tiab]
    OR "decision tree"[tiab] OR "classification tree"[tiab] OR CART[tiab]
    OR "neural network"[tiab] OR "convolutional network"[tiab]
    OR "attention network"[tiab] OR CNN[tiab] OR ResNet[tiab]
    OR "k-nearest neighbor"[tiab] OR "nearest neighbor"[tiab]
    OR "logistic regression"[tiab] OR "multinomial regression"[tiab]
    OR "generalized linear model"[tiab] OR "multivariable model"[tiab]
    OR "linear discriminant"[tiab] OR "quadratic discriminant"[tiab]
    OR "discriminant analysis"[tiab] OR "partial least squares"[tiab]
    OR "PLS-DA"[tiab] OR PARAFAC[tiab] OR chemometric*[tiab]
    OR radiomics[tiab] OR "predictive model"[tiab]
    OR "prediction model"[tiab] OR "diagnostic model"[tiab]
    OR "classification model"[tiab] OR "classification algorithm"[tiab]
    OR "statistical classifier"[tiab] OR "clinical predictor"[tiab]
    OR ("statistical"[tiab] AND "prediction"[tiab])
    OR ("multivariate"[tiab] AND "classification"[tiab])
    OR ("gene signature"[tiab] AND (classif*[tiab] OR diagnos*[tiab]))
)'''

NAMED_PAIR = '''(
    (dengue[tiab] AND (chikungunya[tiab] OR zika[tiab]))
    OR (chikungunya[tiab] AND zika[tiab])
    OR ((COVID-19[tiab] OR SARS-CoV-2[tiab]) AND
        (influenza[tiab] OR RSV[tiab] OR "respiratory syncytial"[tiab]
        OR rhinovirus[tiab] OR adenovirus[tiab] OR parainfluenza[tiab]
        OR metapneumovirus[tiab] OR coronavirus[tiab]))
    OR (influenza[tiab] AND
        (RSV[tiab] OR "respiratory syncytial"[tiab] OR rhinovirus[tiab]
        OR parainfluenza[tiab] OR metapneumovirus[tiab]
        OR adenovirus[tiab]))
    OR (RSV[tiab] AND (rhinovirus[tiab] OR parainfluenza[tiab]
        OR adenovirus[tiab] OR metapneumovirus[tiab]))
    OR (metapneumovirus[tiab] AND parainfluenza[tiab])
    OR (HIV[tiab] AND ("hepatitis B"[tiab] OR "hepatitis C"[tiab]))
    OR ("hepatitis B"[tiab] AND "hepatitis C"[tiab])
    OR (rotavirus[tiab] AND norovirus[tiab])
    OR (enterovirus[tiab] AND adenovirus[tiab])
    OR ("influenza A"[tiab] AND "influenza B"[tiab])
    OR "influenza A and B"[tiab] OR "influenza type A and B"[tiab]
    OR "A and B influenza"[tiab]
)'''

MULTIVIRUS_PANEL = '''(
    "multiple viruses"[tiab] OR "different viruses"[tiab]
    OR "various viruses"[tiab] OR "several viruses"[tiab]
    OR "multiple viral"[tiab] OR "multiple respiratory viruses"[tiab]
    OR "respiratory viruses"[tiab] OR "respiratory pathogens"[tiab]
    OR "viral pathogens"[tiab] OR "viral species"[tiab]
    OR "virus species"[tiab] OR "virus classes"[tiab]
    OR "viral classes"[tiab] OR "viral identities"[tiab]
    OR "virus types"[tiab] OR "multi-virus"[tiab]
    OR "multi-pathogen"[tiab] OR multipathogen[tiab]
    OR "multiclass"[tiab] OR "multi-class"[tiab]
    OR "multilabel"[tiab] OR "multi-label"[tiab]
    OR "binary relevance"[tiab] OR "pathogen-specific"[tiab]
    OR "pathogen identification"[tiab] OR "pathogen prediction"[tiab]
    OR "other viral"[tiab] OR "other respiratory viral"[tiab]
    OR "other respiratory viruses"[tiab] OR "non-COVID viral"[tiab]
    OR "viral co-infections"[tiab] OR "virus mixtures"[tiab]
    OR "viral mixtures"[tiab] OR "mixed infections"[tiab]
    OR "infection conditions"[tiab] OR "virus-by-time"[tiab]
    OR "virus images"[tiab] OR "virus classification"[tiab]
    OR "virus identification"[tiab] OR "virus recognition"[tiab]
    OR "virus texture"[tiab] OR "TEM virus"[tiab]
    OR ("electron microscopy"[tiab] AND "virus"[tiab] AND classif*[tiab])
    OR ("Raman"[tiab] AND "viruses"[tiab] AND classif*[tiab])
    OR ("fluorescence"[tiab] AND "viruses"[tiab] AND discriminat*[tiab])
)'''

TASK = '''(
    classif*[tiab] OR discriminat*[tiab] OR differentiat*[tiab]
    OR distinguish*[tiab] OR identif*[tiab] OR diagnos*[tiab]
    OR "virus recognition"[tiab] OR "viral recognition"[tiab]
    OR "pathogen prediction"[tiab] OR "pathogen-specific prediction"[tiab]
    OR "viral identity prediction"[tiab] OR "detect and identify"[tiab]
    OR "diagnostic model"[tiab] OR "classification model"[tiab]
    OR "infection condition classification"[tiab]
)'''

DATERANGE = ('%04d/%02d/%02d:%04d/%02d/%02d[dp]' %
    (DATE_FROM.year, DATE_FROM.month, DATE_FROM.day,
     DATE_TO.year, DATE_TO.month, DATE_TO.day))

Q1 = '(' + VIRUS + ' AND ' + METHOD + ')'
Q2 = '(' + Q1 + ' AND (' + NAMED_PAIR + ' OR ' + MULTIVIRUS_PANEL + '))'
Q3 = '(' + Q2 + ' AND ' + TASK + ')'
EXCLUDE_TYPES = '(review[pt] OR meta-analysis[pt] OR editorial[pt] OR comment[pt] OR letter[pt])'
Q4 = '(' + Q3 + ' NOT ' + EXCLUDE_TYPES + ')'

def digest(value):
    return hashlib.sha256(value.encode('utf-8')).hexdigest()[:24]

def cache_file(prefix, key):
    return CACHE / (prefix + '_' + digest(key) + '.json')

def cached_get(path):
    if USE_CACHE and path.exists():
        try:
            return json.loads(path.read_text(encoding='utf-8'))
        except (OSError, ValueError):
            pass
    return None

def cached_put(path, data):
    temp = path.with_suffix('.tmp')
    temp.write_text(json.dumps(data, ensure_ascii=False), encoding='utf-8')
    temp.replace(path)

def ncbi_request(endpoint, params, expect_xml=False):
    args = dict(params)
    args.update({'email': EMAIL, 'tool': 'ViralIdentityThreeStageReview'})
    if API_KEY:
        args['api_key'] = API_KEY
    request_data = urlencode(args).encode('utf-8')
    for attempt in range(1, RETRIES + 1):
        try:
            req = Request(NCBI + endpoint, data=request_data, headers={
                'User-Agent': 'ViralIdentityEvidenceSearch/1.0 ' + EMAIL,
                'Content-Type': 'application/x-www-form-urlencoded'})
            with urlopen(req, timeout=TIMEOUT_SECONDS) as handle:
                raw = handle.read()
            time.sleep(0.37 if not API_KEY else 0.12)
            if expect_xml:
                return ET.fromstring(raw)
            data = json.loads(raw.decode('utf-8'))
            if endpoint == 'esearch.fcgi':
                data = data.get('esearchresult', {})
                if 'count' not in data:
                    raise ValueError('No ESearch count: ' + str(data)[:200])
            return data
        except (HTTPError, URLError, OSError, TimeoutError,
                socket.timeout, ValueError, ET.ParseError) as exc:
            if attempt >= RETRIES:
                raise RuntimeError(endpoint + ' failed after %d tries: %s'
                                   % (RETRIES, exc)) from exc
            pause = min(32.0, 1.8 ** attempt + random.random())
            print('    NCBI retry %d/%d: %s; %.1fs' %
                  (attempt, RETRIES, str(exc)[:90], pause))
            time.sleep(pause)

def dated_query(q, d1=DATE_FROM, d2=DATE_TO):
    return '(%s) AND %04d/%02d/%02d:%04d/%02d/%02d[dp]' % (
        q, d1.year, d1.month, d1.day, d2.year, d2.month, d2.day)

def esearch_count(q, d1=DATE_FROM, d2=DATE_TO):
    term = dated_query(q, d1, d2)
    path = cache_file('count', term)
    out = cached_get(path)
    if out is None:
        out = ncbi_request('esearch.fcgi', {
            'db': 'pubmed', 'term': term, 'retmode': 'json', 'retmax': 0})
        cached_put(path, out)
    warnings = out.get('warninglist', {})
    return int(out['count']), warnings

def esearch_ids(q, d1=DATE_FROM, d2=DATE_TO):
    term = dated_query(q, d1, d2)
    path = cache_file('all_ids', term)
    saved = cached_get(path)
    if saved is not None and saved.get('complete'):
        return set(saved['ids'])
    n, _ = esearch_count(q, d1, d2)
    if n > 9000:
        if d1 >= d2:
            raise RuntimeError('Over 9000 hits in one publication date: ' + str(d1))
        middle = d1 + timedelta(days=(d2 - d1).days // 2)
        ids = esearch_ids(q, d1, middle) | esearch_ids(
            q, middle + timedelta(days=1), d2)

    else:
        ids = set()
        for offset in range(0, n, ESEARCH_PAGE):
            pg_path = cache_file('page', term + '__' + str(offset))
            page = cached_get(pg_path)
            if page is None:
                page = ncbi_request('esearch.fcgi', {
                    'db': 'pubmed', 'term': term, 'retmode': 'json',
                    'retstart': offset, 'retmax': ESEARCH_PAGE, 'sort': 'pub_date'})
                if int(page['count']) != n:
                    raise RuntimeError('PubMed count changed during pagination; rerun.')
                cached_put(pg_path, page)
            ids.update(str(v) for v in page.get('idlist', []) if str(v).isdigit())
        if len(ids) != n:
            raise RuntimeError('Incomplete ESearch page retrieval: %d / %d' %
                               (len(ids), n))
    cached_put(path, {'complete': True, 'ids': sorted(ids)})
    return ids

def flattened(node):
    return ' '.join(''.join(node.itertext()).split()) if node is not None else ''

def parse_article(item):
    citation = item.find('MedlineCitation')
    if citation is None:
        return None
    pmid = flattened(citation.find('PMID'))
    article = citation.find('Article')
    if not pmid or article is None:
        return None
    abstract = ' '.join(flattened(x) for x in article.findall(
        './Abstract/AbstractText'))
    if not abstract:
        abstract = ' '.join(flattened(x) for x in item.findall(
            './MedlineCitation/OtherAbstract/AbstractText'))
    year = (article.findtext('./ArticleDate/Year') or
            article.findtext('./Journal/JournalIssue/PubDate/Year') or '')
    if not year:
        stamp = article.findtext('./Journal/JournalIssue/PubDate/MedlineDate') or ''
        m = re.search(r'\b(19|20)\d{2}\b', stamp)
        year = m.group(0) if m else ''
    doi = ''
    for x in item.findall('./PubmedData/ArticleIdList/ArticleId'):
        if x.attrib.get('IdType') == 'doi':
            doi = (x.text or '').strip()
            break
    types = '; '.join(flattened(x) for x in article.findall(
        './PublicationTypeList/PublicationType'))
    return {
        'PMID': pmid,
        'Title': flattened(article.find('ArticleTitle')),
        'Abstract': abstract,
        'Year': year,
        'Journal': flattened(article.find('./Journal/Title')),
        'DOI': doi,
        'Publication_Type': types,
        'PubMed_URL': 'https://pubmed.ncbi.nlm.nih.gov/' + pmid + '/'
    }

def fetch_metadata(ids):
    ordered = sorted(set(ids), key=int)
    results = {}
    for offset in range(0, len(ordered), EFETCH_BATCH):
        batch = ordered[offset:offset + EFETCH_BATCH]
        key = ','.join(batch)
        p = cache_file('efetch', key)
        records = cached_get(p)
        if records is None:
            root = ncbi_request('efetch.fcgi', {
                'db': 'pubmed', 'id': key, 'retmode': 'xml'}, expect_xml=True)
            records = []
            for item in root.findall('./PubmedArticle'):
                parsed = parse_article(item)
                if parsed:
                    records.append(parsed)
            cached_put(p, records)
        for record in records:
            results[record['PMID']] = record
        if (offset // EFETCH_BATCH) % 4 == 0 or offset + EFETCH_BATCH >= len(ordered):
            print('    Abstract metadata: %d/%d' %
                  (min(offset + EFETCH_BATCH, len(ordered)), len(ordered)))

    missing = sorted(set(ordered) - set(results), key=int)
    if missing:
        print('    Missing metadata after batch fetch:', len(missing))
        for pmid in missing[:60]:
            try:
                root = ncbi_request('efetch.fcgi', {
                    'db': 'pubmed', 'id': pmid, 'retmode': 'xml'}, expect_xml=True)
                for item in root.findall('./PubmedArticle'):
                    parsed = parse_article(item)
                    if parsed:
                        results[parsed['PMID']] = parsed
            except RuntimeError:
                pass
    return list(results.values()), sorted(set(ordered) - set(results), key=int)

def load_csv(path):
    with open(str(path), encoding='utf-8-sig', newline='') as f:
        return [dict(r) for r in csv.DictReader(f)]

def write_csv(path, rows, columns=None):
    rows = list(rows)
    if columns is None:
        columns = list(dict.fromkeys(k for r in rows for k in r.keys()))
    with open(str(path), 'w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

HOST_CLASSIFIER_QUERY = '''(
    ("gene expression"[tiab] OR "blood transcriptom"[tiab]
     OR "transcriptional profil"[tiab] OR "host response"[tiab]
     OR "supervised principal component"[tiab])
    AND ("respiratory syncytial"[tiab] OR RSV[tiab])
    AND (rhinovirus[tiab] OR influenza[tiab] OR "non-RSV"[tiab]
         OR "other respiratory virus"[tiab])
    AND (classif*[tiab] OR discriminat*[tiab] OR
         "supervised"[tiab] OR "gene signature"[tiab])
) NOT (review[pt] OR editorial[pt] OR comment[pt] OR letter[pt])'''

def RX(s):
    return re.compile(s, re.I)

PAT = {
    'method': RX(r'machine[ -]learning|deep[ -]learning|artificial intelligence|random forest|gradient boosting|boosted tree|xgboost|catboost|support.vector|\bsvm\b|neural network|convolutional network|\bcnn\b|resnet|densenet|transformer|decision tree|classification tree|\bknn\b|k.nearest.neigh|multinomial logistic|linear discriminant|quadratic discriminant|discriminant analysis|parafac|pls.da|chemometr|cross.validat|supervised|classification model|classifiers?|trained a|trained using|trained on|training set|model training|feature selection'),
    'model': RX(r'model|regression|radiomic|predict|machine|neural|statistical|classification|algorithm|deep[ -]learning|classif'),
    'task': RX(r'classif|discriminat|differentiat|distinguish|diagnos|identif|recognition|predict|etiolog|viral cause|pathogen'),
    'strong_target': RX(r'virus identification|viral identification|virus classification|viral classification|classify viruses|classify different viruses|different viral|virus recognition|pathogen classification|pathogen diagnosis|pathogen prediction|differential diagnosis|discriminat.{0,50}(?:viral|virus|infections)|(?:virus|viral|viruses|pathogen).{0,60}(?:identif|classif|discriminat|differentiat)'),
    'comparison': RX(r'compar|versus|\bvs\.?\b|differentiat|distinguish|discriminat|separat|differential'),
    'host': RX(r'gene expression|transcriptom|transcriptional profil|rna.seq|nanos?tring|multi.omics|t.cell|host response|biosignature|gene signature|blood.based|metabolom|proteom|molecular fingerprint|biomarker|multi.way analysis|microarray'),
    'clinical': RX(r'clinical|laboratory|patient|symptom|blood count|vital sign|admission|infant|pediatric|surveillance|hemogram|epidemiolog|hospital'),
    'imaging': RX(r'computed tomography|\bct\b|chest x.ray|radiograph|radiomics|pneumonia|multimodal|medical imaging|radiologist'),
    'sensing': RX(r'raman|sers|ftir|infrared|fluorescen|biosens|spectroscop|nanopore|parafac|multi.way|spectral|rt.lamp'),
    'particle': RX(r'electron microscop|\btem\b|virus imag|viral imag|virus texture|cytopathic|infected.cell|virus particle|neural spiking|microscop|label.free virus'),
    'panel': RX(r'multi.pathogen|multi.label|multilabel|multi.class|multiclass|respiratory viruses|other viral|multiple viral|multi.virus|several viruses|different viruses|viral species|virus species|virus classes|virus identification|virus recognition|multiple pathogens|mixed infection|influenza a/b|influenza types? a and b|pathogen.specific|virus.by.time|infection conditions|virus.time|\d{1,2} (?:viral|virus|pathogen) (?:classes|categories|species|types)'),
    'bad_target': RX(r'mortalit|hospitalization|hospitalisation|\bicu\b|risk factors?|severity|prognos|length of stay|vaccin|antiviral treat|therap|mutation|forecast|prevalence|antibiotic|proteins? localiz|epitope|infectivity|outbreak forecast|structural protein|microrna|geograph|sequence taxonom|phylogen|genome classif'),
    'perf': RX(r'\bauc\b|auroc|roc.auc|accuracy|cross.validat|sensitivity|specificity|f1.score|precision|recall|test cohort|held.out'),
}

VIRAL = {
 'SARS-CoV-2': RX(r'covid[ -]?19|sars[ -]?cov[ -]?2'),
 'Influenza A': RX(r'influenza (?:type )?a\b|\bh1n1\b|\bh3n2\b'),
 'Influenza B': RX(r'influenza (?:type )?b\b'),
 'Influenza': RX(r'\binfluenza\b'),
 'RSV': RX(r'\brsv\b|respiratory syncytial'),
 'Dengue': RX(r'dengue|\bdenv\b'),
 'Chikungunya': RX(r'chikungunya|\bchikv\b'),
 'Zika': RX(r'\bzika\b|\bzikv\b'),
 'Rhinovirus': RX(r'rhinovirus'),
 'Adenovirus': RX(r'adenovirus'),
 'Parainfluenza': RX(r'parainfluenza|\bpiv3\b'),
 'Metapneumovirus': RX(r'metapneumovirus|\bhmpv\b'),
 'Ebola': RX(r'\bebola\b|\bebov\b'),
 'Coronavirus': RX(r'coronavirus|\bhcov\b'),
 'Enterovirus': RX(r'enterovirus'),
 'Rotavirus': RX(r'rotavirus'),
 'Norovirus': RX(r'norovirus'),
 'HIV': RX(r'\bhiv\b'),
 'HBV': RX(r'hepatitis b|\bhbv\b'),
 'HCV': RX(r'hepatitis c|\bhcv\b'),
 'Herpesvirus': RX(r'herpesvirus'),
 'Poliovirus': RX(r'poliovirus')
}
INFLUENZA_AB = RX(r'\binfluenza\s+(?:types?\s+)?a\s*(?:and|/|\+|&|,)\s*b\b|\ba\s+and\s+b\s+influenza\b')
POOL_VIRAL = RX(r'other viral|non.covid viral|other respiratory virus|other viruses|pooled viral|non.rsv respiratory')
COUNT_CLASSES = RX(r'\b\d{1,2}\s*(?:virus|viral|pathogen)\s*(?:species|types?|classes|categories|labels|targets)\b')
OUTCOME_VERBS = RX(r'classif|discriminat|differentiat|distinguish|identif|diagnos')
VIRAL_WORDS = RX(r'virus|viruses|viral|pathogen|influenza|dengue|chikungunya|covid|\brsv\b|rhinovirus')
REVIEW_TYPES = RX(r'\breview\b|meta.analysis|editorial|\bletter\b|\bcomment\b|guideline')
TITLE_NORMALIZER = RX(r'[^a-z0-9]+')

CLEARLY_OTHER_TASK = RX(r'sequence taxonom|viral genome classif|genom(?:e|ic) (?:representation|sequence|classif)|alignment.free|metagenomic tool|protein localization|epitope predict|antiviral efficacy|antibiotic prescri|cost.consequence|cost.effect|cost.benefit|mutation predict|forecast|seroprevalence|clinical burden|incidence of|protein sequenc|subcellular|global burden|hospital trajector|risk factors? for|prognostic difference|predictors? of (?:mortality|severity|admission|outcome)')
RISK_NOT_IDENTITY = RX(r'risk factors?|disease severity|prognos|mortality|intensive care|\bicu\b|vaccine|prevalence|epidemiolog|clinical outcome|hospitali|antiviral|treatment|forecast')

RULE_NAMES = [
 'Sensing numerical virus classes',
 'Infected-cell or vector virus classification',
 'Host signature distinguishing viral infections',
 'Multiplex viral spectroscopy/assays',
 'COVID-19 versus other viral infection',
 'Virus particle microscopy classifier',
 'Explicit named-virus comparison with model',
 'Viral imaging diagnostic classifier',
 'Clinical multi-pathogen diagnostic model',
 'Influenza A versus B analytical classification',
 'Host transcriptomic differential signature',
 'Direct supervised virus identification',
 'Deep-learning viral-pneumonia comparator',
]

_VIRUS_ORDER = list(VIRAL.keys())
_VIRUS_COMBINED = re.compile('|'.join(
    '(?P<V%d>%s)' % (i, VIRAL[name].pattern)
    for i, name in enumerate(_VIRUS_ORDER)), re.I)

def virus_names(text):
    found = set()
    for match in _VIRUS_COMBINED.finditer(text):
        if match.lastgroup:
            found.add(_VIRUS_ORDER[int(match.lastgroup[1:])])
    if INFLUENZA_AB.search(text):
        found.update({'Influenza A', 'Influenza B'})
    if 'Influenza A' in found or 'Influenza B' in found:
        found.discard('Influenza')
    if 'SARS-CoV-2' in found:
        found.discard('Coronavirus')
    return found

def features(r):
    title = str(r.get('Title') or '')[:1100]
    abstract = str(r.get('Abstract') or '')[:10000]
    text = title+' '+abstract
    f = {k: bool(rx.search(text)) for k,rx in PAT.items()}
    ft = {k: bool(PAT[k].search(title)) for k in ('task','panel','strong_target','comparison','imaging','particle')}
    viruses = virus_names(text)
    named_two = len(viruses)>=2
    pair_title = len(virus_names(title))>=2
    pool_viral = bool(POOL_VIRAL.search(text))
    classcount = bool(COUNT_CLASSES.search(text))
    endpoint_sentence = False
    excerpt = ''
    if f['task'] and (named_two or f['panel'] or pool_viral):
        for s in re.split(r'[.!?;]\s+', abstract[:5000]):
            s = s[:750]
            if OUTCOME_VERBS.search(s) and VIRAL_WORDS.search(s):
                endpoint_sentence = True
                excerpt = s[:430]
                if f['method']:
                    break
    endpoint_title = ft['task'] and (pair_title or ft['panel'] or ft['strong_target'])
    title_virus_image = ft['particle'] and ('virus' in title.lower() or 'viral' in title.lower() or 'tem ' in title.lower())
    rule = {}
    rule[RULE_NAMES[0]] = f['sensing'] and (classcount or ft['panel']) and f['method'] and f['task']
    rule[RULE_NAMES[1]] = f['particle'] and f['panel'] and f['method'] and f['strong_target']
    rule[RULE_NAMES[2]] = f['host'] and named_two and f['task'] and endpoint_sentence and f['comparison']
    rule[RULE_NAMES[3]] = f['sensing'] and f['panel'] and f['method'] and endpoint_sentence
    rule[RULE_NAMES[4]] = pool_viral and f['method'] and f['task'] and (endpoint_title or ft['strong_target'] or ft['comparison'])
    rule[RULE_NAMES[5]] = f['particle'] and (title_virus_image or ft['strong_target']) and f['model'] and f['task']
    rule[RULE_NAMES[6]] = ft['comparison'] and pair_title and (f['method'] or f['model'])
    rule[RULE_NAMES[7]] = f['imaging'] and named_two and f['method'] and (endpoint_title or ft['strong_target'] or endpoint_sentence)
    rule[RULE_NAMES[8]] = f['clinical'] and named_two and f['method'] and f['task'] and f['panel']
    rule[RULE_NAMES[9]] = f['sensing'] and named_two and f['method'] and f['task']
    rule[RULE_NAMES[10]] = f['host'] and named_two and f['task'] and f['method']
    rule[RULE_NAMES[11]] = f['strong_target'] and (named_two or classcount or f['panel']) and f['method'] and f['task']
    rule[RULE_NAMES[12]] = f['imaging'] and f['method'] and f['task'] and f['panel'] and ft['imaging'] and ft['comparison']
    names = [name for name in RULE_NAMES if rule[name]]
    is_nonoriginal = bool(REVIEW_TYPES.search(str(r.get('Publication_Type') or '')))
    likely_other_task = bool(CLEARLY_OTHER_TASK.search(title) or
                           (RISK_NOT_IDENTITY.search(title) and not endpoint_sentence))
    try:
        year = int(str(r.get('Year') or '')[:4])
    except ValueError:
        year = 0
    year_flag = bool(year and (year<2012 or year>2026))
    reason = ('; '.join(names) if names else 'No sufficiently specific general viral-identity rule')
    if year_flag:
        reason += '; year outside requested window (verify early electronic publication)'
    if f['bad_target'] and not endpoint_sentence:
        reason += '; possible non-identity prediction target — verify'
    if likely_other_task:
        reason += '; high-confidence different prediction target in title — check reserve'

    score = (8*int(endpoint_title) + 5*int(endpoint_sentence) +
             4*int(pair_title) + 3*int(named_two) +
             2*int(f['method']) + 2*int(ft['strong_target']) +
             2*int(f['perf']) - 3*int(f['bad_target'] and not endpoint_sentence))
    evidence = dict(r)
    evidence.update({
        'Detected_Viral_Identities': '; '.join(sorted(viruses)),
        'Named_Viruses_Count': len(viruses),
        'Method_Term_Evidence': int(f['method']),
        'Viral_Label_Target_Sentence': excerpt,
        'Pooled_Virus_or_Panel_Signal': int(pool_viral or f['panel'] or classcount),
        'Outcome_Target_Evidence': int(endpoint_title or endpoint_sentence),
        'General_Rules_Matched': '; '.join(names),
        'Rules_Passed': len(names),
        'Auto_Evidence_Score': score,
        'Possible_OffTarget_Outcome': int(f['bad_target']),
        'Likely_Unrelated_Prediction_Target': int(likely_other_task),
        'NonOriginal_Review': int(is_nonoriginal),
        'Publication_Year_Check': int(year_flag),
        'Automatic_Screening_Reason': reason,
        'Automatic_Tier': ('X Non-original/date check' if is_nonoriginal or year_flag else
                           'D Likely off-target — retained for verification' if likely_other_task else
                           'A Provisional: general rules matched' if names else
                           'C Uncertain or lower evidence — not excluded'),
    })
    return evidence

def normalize_doi(s):
    s = str(s or '').lower().strip()
    for prefix in ('https://doi.org/','http://doi.org/','doi:'):
        if s.startswith(prefix): s=s[len(prefix):]
    return s

def normalize_title(s):
    return TITLE_NORMALIZER.sub(' ',str(s or '').lower()).strip()

def publication_score(r):
    text = (str(r.get('Journal',''))+' '+str(r.get('Publication_Type',''))).lower()
    return (0 if 'medrxiv' in text or 'biorxiv' in text or 'preprint' in text else 10,
            1 if r.get('DOI') else 0, len(str(r.get('Abstract',''))))

def report(label, current, previous=None, prefix='  '):
    if previous is None:
        print('%s%-64s %8s' % (prefix, label, format(current, ',')))
        return
    change = previous - current
    percentage = 100.0 * change / previous if previous else 0.0
    print('%s%-52s %9s   removed %9s  (%6.2f%%)' % (
        prefix, label, format(current, ','), format(change, ','), percentage))

def run_workflow():
    OUTPUT.mkdir(parents=True, exist_ok=True)
    CACHE.mkdir(parents=True, exist_ok=True)
    print('=' * 90)
    print('HIERARCHICAL PUBMED SEARCH AND VIRAL-IDENTITY SCREENING')
    print('Publication window: %s to %s inclusive' % (DATE_FROM.isoformat(), DATE_TO.isoformat()))
    print('Search mode: general scientific terms; automatic eligibility-rule screening')
    print('=' * 90)

    stages = [
        ('S1', 'Viral infection + ML / statistical modeling', Q1),
        ('S2', 'Add at least two viruses or viral-panel terms', Q2),
        ('S3', 'Add classification / identification outcome terms', Q3),
        ('S4', 'Remove review and other non-original publication types', Q4),
    ]
    query_rows = [
        {'Stage': code, 'Criterion': label, 'Exact_PubMed_Query': dated_query(q)}
        for code, label, q in stages
    ]
    query_rows.append({
        'Stage': 'H1', 'Criterion': 'Supplementary host-expression classifiers',
        'Exact_PubMed_Query': dated_query(HOST_CLASSIFIER_QUERY)
    })
    write_csv(OUTPUT / '01_EXACT_PUBMED_QUERIES.csv', query_rows,
              ['Stage', 'Criterion', 'Exact_PubMed_Query'])
    (OUTPUT / '01_EXACT_PUBMED_QUERIES.txt').write_text(
        '\n\n'.join(x['Stage'] + ' ' + x['Criterion'] + '\n' +
                    x['Exact_PubMed_Query'] for x in query_rows),
        encoding='utf-8')

    print('\nSTEP 1 / 8 — PUBMED SEARCH COUNTS')
    search_rows = []
    count_map = {}
    prior = None
    for code, label, query in stages:
        count, warning = esearch_count(query)
        count_map[code] = count
        removed = prior - count if prior is not None else ''
        pct = (100.0 * removed / prior) if prior else ''
        search_rows.append({
            'Stage': code, 'Criterion': label, 'Count': count,
            'Removed_From_Previous': removed,
            'Reduction_Percent': round(pct, 3) if pct != '' else '',
            'Warnings': json.dumps(warning, ensure_ascii=False) if warning else ''
        })
        report(code + ' ' + label, count, prior)
        if warning:
            print('    PubMed query warning saved to 02_SEARCH_COUNTS.csv')
            if warning.get('quotedphrasesnotfound'):
                print('    Unrecognized quoted phrases:', ', '.join(warning['quotedphrasesnotfound']))
        prior = count
    write_csv(OUTPUT / '02_SEARCH_COUNTS.csv', search_rows,
              ['Stage', 'Criterion', 'Count', 'Removed_From_Previous',
               'Reduction_Percent', 'Warnings'])

    print('\nSTEP 2 / 8 — DOWNLOAD THE S4 RECORDS')
    direct_ids = esearch_ids(Q4)
    print('  Unique direct PMIDs:', format(len(direct_ids), ','))
    if len(direct_ids) != count_map['S4']:
        print('  Date-window index update: unique ID total differs from cached S4 count.')
    write_csv(OUTPUT / '03_DIRECT_SEARCH_PMIDS.csv',
              [{'PMID': v} for v in sorted(direct_ids, key=int)], ['PMID'])
    direct_records, direct_missing = fetch_metadata(direct_ids)
    by_pmid = {str(r['PMID']): r for r in direct_records}
    report('Successfully downloaded direct records', len(by_pmid), len(direct_ids))
    print('  Missing direct metadata:', len(direct_missing))

    print('\nSTEP 3 / 8 — SUPPLEMENTARY HOST-EXPRESSION SEARCH')
    host_count, host_warning = esearch_count(HOST_CLASSIFIER_QUERY)
    host_ids = esearch_ids(HOST_CLASSIFIER_QUERY)
    additional_ids = host_ids - set(by_pmid)
    added, host_missing = fetch_metadata(additional_ids) if additional_ids else ([], [])
    by_pmid.update({str(r['PMID']): r for r in added})
    print('  Supplementary query hits:', format(host_count, ','))
    print('  Supplementary query unique IDs:', format(len(host_ids), ','))
    print('  IDs not already in downloaded direct records:', format(len(additional_ids), ','))
    print('  Additional records successfully downloaded:', format(len(added), ','))
    print('  Combined unique records:', format(len(by_pmid), ','))
    if host_warning:
        print('  Host-expression PubMed warnings saved')
    write_csv(OUTPUT / '04_SUPPLEMENTARY_SEARCH_PMIDS.csv',
              [{'PMID': x, 'Added_After_Direct_Search': int(x in additional_ids)}
               for x in sorted(host_ids, key=int)],
              ['PMID', 'Added_After_Direct_Search'])
    write_csv(OUTPUT / '05_MISSING_METADATA_PMIDS.csv',
              [{'PMID': x, 'Source': 'Direct'} for x in direct_missing] +
              [{'PMID': x, 'Source': 'Supplementary'} for x in host_missing],
              ['PMID', 'Source'])

    print('\nSTEP 4 / 8 — APPLY 13 GENERAL SCIENTIFIC SCREENING RULES')
    records = sorted(by_pmid.values(), key=lambda r: int(r['PMID']))
    temp_dir = OUTPUT / '_screening_chunks'
    temp_dir.mkdir(parents=True, exist_ok=True)
    chunk_files = []
    for start in range(0, len(records), 1200):
        portion = [features(r) for r in records[start:start + 1200]]
        path = temp_dir / ('screened_%05d.csv' % start)
        write_csv(path, portion)
        chunk_files.append(path)
        print('  Screened %s/%s abstracts' % (
            format(min(start + 1200, len(records)), ','), format(len(records), ',')))
    del records
    assessed = []
    numeric = ('Rules_Passed', 'NonOriginal_Review',
               'Publication_Year_Check', 'Likely_Unrelated_Prediction_Target',
               'Auto_Evidence_Score')
    for path in chunk_files:
        for row in load_csv(path):
            for field in numeric:
                row[field] = int(row.get(field) or 0)
            assessed.append(row)

    print('\nSTEP 5 / 8 — SEQUENTIAL SCIENTIFIC FILTERING AND DEDUPLICATION')
    pre_filter = len(assessed)
    rule_matches = [r for r in assessed if r['Rules_Passed'] > 0]
    after_original = [r for r in rule_matches if not r['NonOriginal_Review']]
    after_year = [r for r in after_original if not r['Publication_Year_Check']]
    accepted_versions = [r for r in after_year
                         if not r['Likely_Unrelated_Prediction_Target']]
    print('  General rules evaluated:', len(RULE_NAMES))
    report('All downloaded records', pre_filter)
    report('Match one or more viral-identity rules', len(rule_matches), pre_filter)
    report('Exclude non-original publication types', len(after_original), len(rule_matches))
    report('Remove records outside publication-year window', len(after_year), len(after_original))
    report('Flag unrelated prediction targets', len(accepted_versions), len(after_year))

    groups = defaultdict(list)
    for record in assessed:
        doi = normalize_doi(record.get('DOI'))
        title = normalize_title(record.get('Title'))
        key = ('title', title) if len(title) > 35 else (
              ('doi', doi) if doi else ('pmid', record['PMID']))
        groups[key].append(record)
    qualified_ids = {record['PMID'] for record in accepted_versions}
    for versions in groups.values():
        eligible_versions = [record for record in versions if record['PMID'] in qualified_ids]
        preferred = max(eligible_versions if eligible_versions else versions, key=publication_score)
        for record in versions:
            record['Possible_Duplicate_PMIDs'] = '; '.join(
                sorted(x['PMID'] for x in versions if x['PMID'] != record['PMID']))
            record['Preferred_Version'] = int(record is preferred)
    selected = [r for r in accepted_versions if r['Preferred_Version']]
    selected.sort(key=lambda r: (
        -r['Auto_Evidence_Score'], -r['Rules_Passed'], str(r.get('Title', ''))))
    report('Keep preferred publication version', len(selected), len(accepted_versions))
    print('  AUTOMATED PROVISIONAL SELECTION:', format(len(selected), ','))

    selected_ids = {r['PMID'] for r in selected}
    reserve = [r for r in assessed if r['PMID'] not in selected_ids]
    duplicate_flags = [r for r in assessed if r['Possible_Duplicate_PMIDs']]
    target_flags = [r for r in assessed if r['Likely_Unrelated_Prediction_Target']]
    review_flags = [r for r in assessed if r['NonOriginal_Review']]
    date_flags = [r for r in assessed if r['Publication_Year_Check']]
    print('  All-record flags (may overlap):')
    print('    Non-original publication types:', len(review_flags))
    print('    Year outside 2012–2026:', len(date_flags))
    print('    Likely unrelated prediction target:', len(target_flags))
    print('    Potential duplicate-version rows:', len(duplicate_flags))
    print('    Records in retained reserve:', len(reserve))

    print('\nSTEP 6 / 8 — INDIVIDUAL RULE CONTRIBUTIONS')
    rule_rows = []
    union = set()
    for number, name in enumerate(RULE_NAMES, 1):
        matches = {r['PMID'] for r in accepted_versions
                   if name in r['General_Rules_Matched'].split('; ')}
        new = matches - union
        union |= matches
        item = {
            'Rule_Number': number, 'General_Rule': name,
            'Matched_Records': len(matches),
            'New_Unique_Records': len(new),
            'Cumulative_Unique_Records': len(union)
        }
        rule_rows.append(item)
        print('  R%02d %-53s matched %5s; new %5s; cumulative %5s' % (
            number, name[:53], format(len(matches), ','),
            format(len(new), ','), format(len(union), ',')))
    assert union == {r['PMID'] for r in accepted_versions}
    write_csv(OUTPUT / '07_RULE_CONTRIBUTION_COUNTS.csv', rule_rows,
              ['Rule_Number', 'General_Rule', 'Matched_Records',
               'New_Unique_Records', 'Cumulative_Unique_Records'])

    print('\nSTEP 7 / 8 — EXPORT RESULTS AND ALL FILTERING COUNTS')
    write_csv(OUTPUT / '06_ALL_RECORDS_WITH_GENERAL_RULE_EVIDENCE.csv', assessed)
    write_csv(OUTPUT / '08_SELECTED_PROVISIONAL_STUDIES.csv', selected)
    write_csv(OUTPUT / '09_MATCHED_ALL_VERSIONS_BEFORE_DEDUP.csv', accepted_versions)
    write_csv(OUTPUT / '10_NOT_SELECTED_PRESERVED.csv', reserve)
    write_csv(OUTPUT / '11_DUPLICATE_VERSION_FLAGS.csv', duplicate_flags)
    write_csv(OUTPUT / '12_OFFTARGET_FLAGS_REVIEW.csv', target_flags)
    write_csv(OUTPUT / '13_FIRST_120_SCREENING_ONLY.csv', selected[:FIRST_REVIEW_BATCH])

    checklist_columns = [
        'PMID', 'Title', 'PubMed_URL', 'General_Rules_Matched',
        'Detected_Viral_Identities', 'Viral_Label_Target_Sentence',
        'Supervised_Model_YES_NO', 'Two_Distinct_Viruses_YES_NO',
        'Virus_Identity_Is_Prediction_Target_YES_NO', 'Original_Paper_YES_NO',
        'Not_Duplicate_YES_NO', 'Final_Decision', 'Full_Text_Notes'
    ]
    checklist = OUTPUT / '14_FULLTEXT_CHECKLIST_FOR_VERIFICATION.csv'
    if not checklist.exists():
        write_csv(checklist,
                  [{k: r.get(k, '') for k in checklist_columns} for r in selected],
                  checklist_columns)
    else:
        print('  Existing full-text checklist preserved')

    summary = []
    def add(code, criterion, count, previous=None, group='Sequential'):
        removed = previous - count if previous is not None else ''
        percent = round(100.0 * removed / previous, 3) if previous else ''
        summary.append({
            'Stage': code, 'Criterion': criterion, 'Count': count,
            'Removed_From_Previous': removed,
            'Reduction_Percent': percent, 'Sequence': group
        })

    for i, (code, label, _) in enumerate(stages):
        previous = count_map[stages[i - 1][0]] if i else None
        add(code, label, count_map[code], previous)
    add('S5', 'Direct PMID IDs identified', len(direct_ids), count_map['S4'])
    add('S6', 'Direct records with downloaded metadata', len(direct_records), len(direct_ids))
    add('H1', 'Supplementary query hits', host_count, None, 'Supplementary')
    add('H2', 'Additional unique IDs from supplementary search', len(additional_ids), None, 'Supplementary')
    add('H3', 'Additional supplementary records downloaded', len(added), len(additional_ids), 'Supplementary')
    add('S7', 'Combined unique records with metadata', len(assessed), None, 'Merged')
    add('S8', 'Meet at least one general scientific screening rule', len(rule_matches), len(assessed))
    add('S9', 'After non-original publication-type screening', len(after_original), len(rule_matches))
    add('S10', 'After publication-year check', len(after_year), len(after_original))
    add('S11', 'After prediction-target screen', len(accepted_versions), len(after_year))
    add('S12', 'Unique preferred publication versions', len(selected), len(accepted_versions))
    add('R', 'All other records preserved for verification', len(reserve), None, 'Reserve')
    add('F1', 'Flagged non-original records (overlapping)', len(review_flags), None, 'Flags')
    add('F2', 'Flagged publication year (overlapping)', len(date_flags), None, 'Flags')
    add('F3', 'Flagged unrelated outcome (overlapping)', len(target_flags), None, 'Flags')
    add('F4', 'Possible duplicate-version records (overlapping)', len(duplicate_flags), None, 'Flags')
    write_csv(OUTPUT / '00_STEPWISE_COUNTS_START_TO_FINISH.csv', summary,
              ['Stage', 'Criterion', 'Count', 'Removed_From_Previous',
               'Reduction_Percent', 'Sequence'])

    print('\nSTEP 8 / 8 — COMPLETE FILTERING FUNNEL')
    for row in summary:
        if row['Sequence'] not in ('Flags', 'Reserve'):
            count = format(row['Count'], ',')
            removed = row['Removed_From_Previous']
            pct = row['Reduction_Percent']
            difference = ('removed %s (%s%%)' % (format(removed, ','), pct)
                          if removed != '' else 'supplementary / merged count')
            print('  %-4s %-57s %9s  %s' % (
                row['Stage'], row['Criterion'][:57], count, difference))
    print('\n  ADDITIONAL FLAGS AND RESERVE')
    for row in summary:
        if row['Sequence'] in ('Flags', 'Reserve'):
            print('  %-4s %-57s %9s' % (
                row['Stage'], row['Criterion'][:57], format(row['Count'], ',')))
    print('  Total from initial PubMed search:', format(count_map['S1'], ','))
    print('  Final provisional study count:', format(len(selected), ','))
    print('  Output folder:', OUTPUT.resolve())
    print('  Study list:', OUTPUT / '08_SELECTED_PROVISIONAL_STUDIES.csv')
    print('  Stage counts:', OUTPUT / '00_STEPWISE_COUNTS_START_TO_FINISH.csv')
    print('  All abstracts, rule matches and reserve were exported')
    return {'selected': selected, 'counts': summary, 'all_records': assessed}

if __name__ == '__main__':
    run_workflow()


HIERARCHICAL PUBMED SEARCH AND VIRAL-IDENTITY SCREENING
Publication window: 2012-01-01 to 2026-10-10 inclusive
Search mode: general scientific terms; automatic eligibility-rule screening

STEP 1 / 8 — PUBMED SEARCH COUNTS
  S1 Viral infection + ML / statistical modeling                     75,718
  S2 Add at least two viruses or viral-panel terms        14,714   removed    61,004  ( 80.57%)
    PubMed query warning saved to 02_SEARCH_COUNTS.csv
    Unrecognized quoted phrases: "influenza type A and B"[tiab], "A and B influenza"[tiab], "viral identities"[tiab], "virus-by-time"[tiab], "virus texture"[tiab], "TEM virus"[tiab]
  S3 Add classification / identification outcome terms     9,014   removed     5,700  ( 38.74%)
    PubMed query warning saved to 02_SEARCH_COUNTS.csv
    Unrecognized quoted phrases: "influenza type A and B"[tiab], "A and B influenza"[tiab], "viral identities"[tiab], "virus-by-time"[tiab], "virus texture"[tiab], "TEM virus"[tiab], "pathogen-specific prediction"[tiab

In [2]:
import csv
import re
from pathlib import Path
from collections import Counter

def run_viral_identity_false_positive_filter():
    filename = '08_SELECTED_PROVISIONAL_STUDIES.csv'
    paths = []
    prior_output = globals().get('OUTPUT')
    if prior_output is not None:
        paths.append(Path(prior_output) / filename)
    paths += [Path.cwd() / 'VirusIdentity_Search_2012_2026_10_10' / filename,
              Path.cwd() / filename]
    paths += sorted(Path.cwd().glob('*/' + filename))
    paths += sorted(Path.cwd().glob('*/*/' + filename))
    INPUT = next((p for p in paths if p.is_file()), None)
    if INPUT is None:
        raise FileNotFoundError('Cannot find 08_SELECTED_PROVISIONAL_STUDIES.csv. Put the file beside the notebook or in VirusIdentity_Search_2012_2026_10_10.')
    OUTPUT = INPUT.resolve().parent / 'Viral_Identity_FalsePositive_Screen'
    OUTPUT.mkdir(parents=True, exist_ok=True)

    VIRUS_RX = {
    'SARS-CoV-2':r'\bcovid[ -]?19\b|\bsars[ -]?cov[ -]?2\b',
    'Influenza A':r'\binfluenza\s+(?:virus\s+)?(?:type\s+)?a\b|\bh1n1\b|\bh3n2\b',
    'Influenza B':r'\binfluenza\s+(?:virus\s+)?(?:type\s+)?b\b',
    'Influenza':r'\binfluenza\b',
    'RSV':r'\brsv\b|respiratory syncytial',
    'Dengue':r'\bdengue\b|\bdenv\b',
    'Zika':r'\bzika\b|\bzikv\b',
    'Chikungunya':r'chikungunya|\bchikv\b',
    'Rhinovirus':r'rhinovirus|\bhrv\b',
    'Adenovirus':r'adenovirus',
    'Parainfluenza':r'parainfluenza|\bpiv3\b',
    'Metapneumovirus':r'metapneumovirus|\bhmpv\b',
    'Coronavirus':r'\b(?:hcov[- ]?(?:nl63|oc43|229e|hku1)|seasonal coronaviruses?|coronaviruses?)\b',
    'Enterovirus':r'enterovirus',
    'Rotavirus':r'rotavirus',
    'Norovirus':r'norovirus',
    'Ebola':r'\bebola\b|\bebov\b',
    'HIV':r'\bhiv\b|human immunodeficiency virus',
    'HBV':r'\bhbv\b|hepatitis b(?: virus)?',
    'HCV':r'\bhcv\b|hepatitis c(?: virus)?',
    'Herpesvirus':r'herpesvirus|\bhsv\b',
    'Poliovirus':r'poliovirus',
    'Varicella-zoster':r'varicella.zoster|\bvzv\b',
    'Bovine herpesvirus':r'\bboghv\b|bovine herpesvirus',
    'Infectious bronchitis virus':r'infectious bronchitis virus',
    'Papillomavirus':r'papillomavirus|\bhpv\b',
    'Yellow fever virus':r'yellow fever virus',
    'Mpox':r'mpox|monkeypox',
    }
    VIRUSES = {k:re.compile(v,re.I) for k,v in VIRUS_RX.items()}
    RX = lambda p:re.compile(p,re.I)
    A_B = RX(r'\binfluenza\s+(?:virus\s+)?(?:types?\s+)?a\s*(?:and|/|\+|&|,)\s*b\b|\ba\s+and\s+b\s+influenza\b|\bflu\s*a\s*(?:and|/|\+|&)\s*b\b')
    METHOD_STRONG = RX(r'\b(?:machine[ -]learning|deep[ -]learning|artificial intelligence|supervised|random forests?|xgboost|catboost|lightgbm|gradient boosting|boosted tree|decision tree|classification tree|regression tree|\bCART\b|support.vector|\bSVM\b|convolutional neural|neural network|\bCNN\b|\bkNN\b|k.nearest.neigh|resnet|densenet|transformer|linear discriminant|quadratic discriminant|discriminant analysis|\bPLS.DA\b|parafac|chemometric|multinomial logistic|logistic regression|radiomic|cross.validation|train.test|training data|train(?:ed|ing) (?:a |the |with )?(?:model|classifier|network)|classifier\b|classification model|predictive model|prediction model|diagnostic model|generalized linear model|probit model|multivariate classifier|\bLDA\b|\bQDA\b)')
    METHOD_WEAK = RX(r'\b(model|predict(?:ion|ive)?|classif(?:ier|ication)|algorithm|multivariate|regression|feature selection|deep|clinical predictors?)\b')
    MODEL_OUTCOME = RX(r'\b(classif\w*|differentiat\w*|discriminat\w*|distinguish\w*|identif\w*|detect\w*|diagnos\w*|assign\w*|predict\w*|separat\w*|recogniz\w*)\b')
    STRONG_IDENT = RX(r'\b(?:virus|viral|viruses|pathogen|respiratory virus|infection)[\s\S]{0,70}(?:classif|identif|discriminat|differentiat|recognit)|\b(?:classif|identif|discriminat|differentiat|distinguish|diagnos)[\s\S]{0,75}\b(?:virus|viral|viruses|pathogen|influenza|dengue|zika|chikungunya)\b')
    PAIR_CLAIM = RX(r'\b(?:differentiat\w*|distinguish\w*|discriminat\w*|classif\w*|separat\w*|diagnos\w*|identif\w*|predict\w*)\b')
    POOL = RX(r'other viral (?:infection|pneumonia|respiratory)|other respiratory (?:viruses?|infections?)|non.covid viral|other viruses|pooled viral|non.rsv viral|non.covid respiratory virus')
    MULTI = RX(r'\b(?:multiple viruses|different viruses|multi.virus|multi.pathogen|multiclass|multi.class|multi.label|multilabel|viral panel|virus panels?|\d+\s*(?:virus|viral|pathogen)\s*(?:species|categories|classes|types|labels|targets)|\d+\s*virus\s*(?:image|class)|virus classes|viral categories|pathogen.specific|viral co.infection|virus mixtures|viral mixtures|mixed viral infection)\b')
    GENOMIC_TAX = RX(r'\b(?:taxonom|phylogen|genom(?:e|ic)|metagenom|k.mer|gene sequence|nucleotide sequence|viral sequences?|viral peptide|antiviral peptide|microRNA|micro.rna|epitope|structural protein|protein language model|host range|zoonotic transmi|viral fusion peptide|virus escape|mutation|antigenic drift|phylogeograph)\b')
    HCC = RX(r'\b(?:hepatocellular carcinoma|\bHCC\b|cirrhosis|fibrosis|liver disease|non.alcoholic fatty|nafld|nash|hepatitis progression|transplant(?:ation)?|tumou?r|oncolog|cancer|patient survival|hepatic)\b')
    PUBLIC_HEALTH = RX(r'\b(?:twitter|social media|infodemiol|knowledge attitudes|vaccination|vaccine|overdose|injection drug|antibody development|drug discovery|drug treatment|cost.effect|cost.benefit|economic|resource allocation|prescribing|antibiotic|therapy response|treatment efficacy|informed consent|adherence|screening uptake|refusal|healthcare access|disease surveillance trend|weather|climate forecast|environmental spillover|perception|wearable)\b')
    PROGNOSIS = RX(r'\b(?:severity|severe cases?|risk factors?|risk of|prognos|mortalit|icu|intensive care|mechanical ventilation|length of stay|hospitali[sz]ation|admission|clinical outcome|surviv|hypox|oxygen saturation|exacerbation|long covid|recovery|protection|persistent arthralgia|symptom burden|rehospitali|outcome prediction)\b')
    GENE_DISCOVERY = RX(r'\b(?:differentially expressed genes?|differential expression|gene network|immune pathway|pathway enrichment|interaction network|immune response|transcriptional response|biological mechanisms?|molecular mechanisms?|biomarker discovery|gene set enrichment|single.cell atlas|protein.protein interactions)\b')
    NONORIGINAL = RX(r'\b(?:review|meta.analysis|protocol|editorial|correction|erratum|comment|letter to the editor)\b')
    CONTROL_ONLY = RX(r'\b(?:normal|healthy|negative control|uninfected|non.infected|bacterial pneumonia|non.viral pneumonia|non.viral|no pneumonia|control group|pneumonia and normal|normal and pneumonia)\b')
    IMAGING = RX(r'\b(?:ct|computed tomography|chest x.ray|radiograph|lung imaging|radiomics|magnetic resonance|spectroscop|Raman|SERS|FTIR|fluorescen|electron microscop|virus imag|particle imag|microscop|cytopathic|neural spiking)\b')
    COV_ONLY_TITLE = RX(r'\b(?:covid.19|sars.cov.2|hiv|hepatitis c|hepatitis b|dengue|zika|influenza|\brsv\b)\b')
    HARD_UNRELATED_TITLE = RX(r'\b(?:antimicrobial peptides?|antiviral peptides?|peptide functional|virus fusion peptides?|epitope.based vaccine|protein language model|genome.foundation model|\bviral escape\b|drug discovery|antiviral (?:activity|efficacy)|antibody (?:reactivity|engineering|specificity)|tire identification|breeding site detection|mask classification|sentiment analysis|twitter|social media|microRNA precursors?|cost of (?:HAART|treatment|care)|treatment (?:cost|response)|sex differentiation|outcome[s]? of (?:respiratory|viral)|mortality|oxygen saturation|hospitalization risk|predict(?:ing|ion) (?:of )?(?:severe|mortality|hospitalization|admission|risk|disease severity)|\bhcc\b|hepatocellular carcinoma|liver fibrosis|liver cirrhosis|\bnafld\b|\bNASH\b|protein.protein interaction networks|host dependency factors|predictors of persistent arthralgia|host specificity of (?:coronavirus|virus)|climate|meteorological drivers|vaccination (?:rates|trends|practices)|disease relapse|disease recurrence)\b')



    def virus_names(text):
        found={k for k,v in VIRUSES.items() if v.search(text)}
        if A_B.search(text): found.update(['Influenza A','Influenza B'])
        if 'Influenza A' in found or 'Influenza B' in found: found.discard('Influenza')
        if 'SARS-CoV-2' in found and 'Coronavirus' in found:
            if not RX(r'\b(?:seasonal coronaviruses?|hcov|\b(?:nl63|oc43|229e|hku1)\b)\b').search(text): found.discard('Coronavirus')
        return found

    def title_pair(title):
        return len(virus_names(title))>=2 or bool(POOL.search(title) and COV_ONLY_TITLE.search(title))

    def get_evidence(title,abstract):
        sentences=[s.strip() for s in re.split(r'(?<=[.!?;])\s+|\n+',abstract[:11000]) if len(s.strip())>=12][:60]
        explicit=[]; broad=[]; model=[]
        for i,s in enumerate(sentences):
            names=virus_names(s)
            task=bool(MODEL_OUTCOME.search(s))
            meth=bool(METHOD_STRONG.search(s))
            has_pool=bool(POOL.search(s))
            has_multi=bool(MULTI.search(s))
            explicit_target = bool(STRONG_IDENT.search(s)) or (len(names)>=2 and bool(PAIR_CLAIM.search(s))) or (has_pool and task and bool(COV_ONLY_TITLE.search(s))) or (has_multi and task and bool(re.search(r'virus|pathogen|viral',s,re.I)))
            neg_only = bool(PROGNOSIS.search(s)) and not (len(names)>=2 and task) and not (has_pool and task)
            if task and explicit_target and not neg_only: explicit.append((i,s))
            elif task and (len(names)>=2 or has_pool or has_multi): broad.append((i,s))
            if meth: model.append((i,s))
        return explicit,broad,model

    def assess(row):
        title=str(row.get('Title') or '')
        abstract=str(row.get('Abstract') or '')
        text=title+' '+abstract
        title_names=virus_names(title)
        names=virus_names(text)
        explicit,broad,model=get_evidence(title,abstract)
        model_strong=bool(METHOD_STRONG.search(text))
        model_weak=bool(METHOD_WEAK.search(text))
        pooled=bool(POOL.search(title) and MODEL_OUTCOME.search(title)) or any(POOL.search(x) and MODEL_OUTCOME.search(x) for x in re.split(r'(?<=[.!?;])\s+', abstract[:10000]))
        viral_panel = bool(RX(r'\b(?:multiple viruses|different viruses|multi.virus|multi.pathogen|viral panel|virus panels?|\d+\s*(?:virus|viral|pathogen)\s*(?:species|categories|classes|types|labels|targets|subtypes)|\d+\s*virus\s*(?:image|class)|virus classes|viral categories|pathogen.specific|viral co.infection|virus mixtures|viral mixtures|mixed viral infection|respiratory (?:virus|pathogen) panel|other respiratory viruses)\b').search(text))
        numeric_viral_panel = bool(RX(r'\b(?:[2-9]|[1-9]\d)\s+(?:different\s+|distinct\s+|individual\s+|common\s+)?(?:respiratory\s+|human\s+|avian\s+|viral\s+)?(?:viruses|virus\s+species|viral\s+(?:species|types|categories|classes)|pathogens|pathogen\s+(?:subtypes|categories|targets))\b').search(text))
        multivirus = numeric_viral_panel or viral_panel or bool(MULTI.search(text) and (len(names)>=2 or pooled))
        title_pairs=title_pair(title)
        titles_explicit=bool(MODEL_OUTCOME.search(title) and (title_pairs or (MULTI.search(title) and re.search('virus|viral|pathogen',title,re.I))))
        multi_evidence=(len(names)>=2 or pooled or multivirus)
        identity_evidence=bool(explicit) or titles_explicit
        explicit_abstract=bool(explicit)
        negative=[]; caution=[]; positive=[]
        t=title.lower()
        a=abstract.lower()
        if not abstract.strip(): caution.append('Abstract unavailable')
        if len(names)<2 and not pooled and not multivirus: caution.append('Second viral identity not established in metadata')
        if not model_strong: caution.append('No explicit supervised classifier/method term in metadata')
        if not identity_evidence: caution.append('Actual model output labels not explicit in title/abstract')
        if CONTROL_ONLY.search(text) and len(names)<2 and not pooled and not multivirus:
            if 'covid' in t or 'dengue' in t or 'influenza' in t or 'virus' in t: negative.append('Only one identifiable virus with nonviral controls')
        if GENOMIC_TAX.search(title) and not bool(IMAGING.search(title)) and not (explicit_abstract and RX(r'(clinical|blood|serum|cell imaging|host response|infection conditions|specimen)').search(text)):
            negative.append('Genomic sequence/taxonomy/within-virus prediction')
        if (HCC.search(title) or PUBLIC_HEALTH.search(title)) and not (explicit_abstract and RX(r'(virus infection|viral identity|virus classification|pathogen identification)').search(text)):
            negative.append('Clinical/epidemiologic outcome unrelated to viral identity')
        if PROGNOSIS.search(title) and not explicit_abstract:
            negative.append('Prognosis, severity or risk endpoint without viral classifier')
        if title.startswith('Meta-Analysis') or (NONORIGINAL.search(title) and ('review' in t or 'meta-analysis' in t or 'protocol' in t)):
            negative.append('Review, meta-analysis, or protocol')
        if GENE_DISCOVERY.search(title) and not identity_evidence and not titles_explicit and not explicit_abstract:
            if not model_strong: negative.append('Descriptive molecular comparison without predictive model')
            else: caution.append('Molecular comparison may lack virus-identity prediction')
        genome_sequence_model = bool(RX(r'\b(?:genomic sequence|viral sequence|viral genome|genomic taxonomy|taxonomic classif|k.mer representation|chaos game representation|classification of (?:genomes|sequences)|protein language models?|sequence.based (?:classif|identif)|genome based|genomic features)\b').search(text)) and not RX(r'\b(?:host transcriptom|RNA.seq|blood sample|blood RNA|serum sample|plasma sample|chest CT|computed tomography|microscopy images|electron microscopy|Raman|FTIR|SERS|mass spectrometry)\b').search(title)
        if genome_sequence_model:
            negative.append('Sequence-based viral taxonomy or molecular-entity classification')
        if HARD_UNRELATED_TITLE.search(title) or (GENOMIC_TAX.search(title) and RX(r'\b(?:sequence|genom|taxonom|\bk.mer\b|representation|motif|subtyp|variant)\b').search(title)):
            negative.append('Title identifies a different modeled outcome or molecular entity')
        if len(names)<=1 and not pooled and not viral_panel and RX(r'\b(?:covid|sars.cov.2|influenza|dengue|rsv|hepatitis)\b').search(title):
            if CONTROL_ONLY.search(text) and RX(r'(?:classification|classif|distinguish|detect|diagnos)').search(title):
                negative.append('Apparent single-virus versus nonviral control task')
        if RX(r'\bnon.covid(?:.19)?\b').search(text) and len(names)<=1 and not pooled and not viral_panel and RX(r'\b(?:covid.19|sars.cov.2)\b').search(title):
            negative.append('Modeled COVID-19 versus nonspecific non-COVID/normal controls')
        if not multi_evidence and not identity_evidence:
            negative.append('No evidence of two distinct viral prediction labels')
        if title_pairs: positive.append('Title specifies two viral identities/pooled viral comparator')
        if explicit_abstract: positive.append('Abstract describes viral identification/differentiation')
        if model_strong: positive.append('Supervised classifier/method terminology')
        if multivirus: positive.append('Multi-virus panel terminology')
        if pooled: positive.append('Pooled other-viral comparator')
        if len(names)>=2: positive.append('At least two virus names in metadata')
        if negative and (genome_sequence_model or HARD_UNRELATED_TITLE.search(title) or ('Modeled COVID-19 versus nonspecific non-COVID/normal controls' in negative) or not (explicit_abstract and not (GENOMIC_TAX.search(title) or HCC.search(title) or PUBLIC_HEALTH.search(title)))):
            if 'No evidence of two distinct viral prediction labels' in negative and len(names)>=2: negative.remove('No evidence of two distinct viral prediction labels')
            group='LIKELY_FALSE_POSITIVE'
        elif identity_evidence and model_strong and multi_evidence and not negative:
            if explicit_abstract and (title_pairs or len(names)>=2 or pooled or multivirus): group='HIGH_CONFIDENCE_CANDIDATE'
            else: group='POSSIBLE_CANDIDATE'
        elif model_strong and multi_evidence and not negative:
            group='POSSIBLE_CANDIDATE'
        else:
            group='NEEDS_FULL_TEXT_CHECK'
        single_covid_binary = bool(RX(r'\bnon.covid(?:.19)?\b').search(abstract) and RX(r'\b(?:binary class|binary classification|COVID.19 and non.COVID.19|COVID.19 versus non.COVID.19)\b').search(abstract))
        explicit_other_viral_label = bool(POOL.search(title)) or any(POOL.search(s) and RX(r'\b(?:discriminat|classif|differentiat|distinguish|versus|vs\.?|between)\b').search(s) and RX(r'\b(?:model|classif|diagnos|predict|system|test)\b').search(s) for s in re.split(r'(?<=[.!?;])\s+', abstract[:9000]) if RX(r'\b(?:model|classif|diagnos|predict|system|test)\b').search(s) and RX(r'\b(?:between|versus|vs\.?|other viral)\b').search(s))
        if group in ('HIGH_CONFIDENCE_CANDIDATE','POSSIBLE_CANDIDATE') and single_covid_binary and len(names)<=1 and not explicit_other_viral_label:
            group = 'NEEDS_FULL_TEXT_CHECK'
            caution.append('Binary COVID/non-COVID labels; confirm other viral comparators before inclusion')
        def ex(arr):return (arr[0][1][:420] if arr else '')
        out=dict(row)
        out.update({'Second_Pass_Decision':group,'Candidate_Viral_Labels':'; '.join(sorted(names)),
                    'Named_Virus_Count_Rechecked':len(names), 'Explicit_Title_Virus_Identity_Task':int(titles_explicit),
                    'Viral_Prediction_Abstract_Evidence':ex(explicit), 'Ambiguous_Prediction_Abstract_Evidence':ex(broad),
                    'Method_Evidence_Sentence':ex(model), 'General_ML_Method_Present':int(model_strong),
                    'Two_Viruses_or_Pooled_Panel_Evidence':int(multi_evidence),
                    'Prediction_Target_Evidence':int(identity_evidence),
                    'Flagged_False_Positive_Reasons':' | '.join(negative),
                    'Uncertainty_Reasons':' | '.join(caution),
                    'Positive_Evidence':' | '.join(positive)})
        return out



    def write_csv(path, rows, fields=None):
        rows=list(rows)
        if fields is None:
            fields=list(rows[0]) if rows else ['PMID','Title']
        with open(path,'w',encoding='utf-8-sig',newline='') as handle:
            writer=csv.DictWriter(handle,fieldnames=fields,extrasaction='ignore')
            writer.writeheader()
            writer.writerows(rows)

    with INPUT.open(encoding='utf-8-sig',newline='') as handle:
        originals=list(csv.DictReader(handle))
    required=('PMID','Title','Abstract')
    missing=[col for col in required if not originals or col not in originals[0]]
    if missing:
        raise ValueError('Input CSV lacks required columns: '+', '.join(missing))
    unique={}
    for record in originals:
        pmid=str(record.get('PMID','')).strip()
        if pmid:
            unique[pmid]=record
    records=list(unique.values())
    print('='*83)
    print('SECOND-PASS SUPERVISED VIRAL-IDENTITY ELIGIBILITY FILTER')
    print('Input:',INPUT.resolve())
    print('Output:',OUTPUT.resolve())
    print('='*83)
    print('Stage 0 — Input records:',format(len(originals),','))
    print('Stage 1 — Unique identifiers:',format(len(records),','))

    scored=[]
    for index,record in enumerate(records,1):
        scored.append(assess(record))
        if index % 100==0 or index==len(records):
            print('  Reviewed metadata: {:,}/{:,}'.format(index,len(records)))

    ORDER=('HIGH_CONFIDENCE_CANDIDATE','POSSIBLE_CANDIDATE','NEEDS_FULL_TEXT_CHECK','LIKELY_FALSE_POSITIVE')
    groups={key:[row for row in scored if row['Second_Pass_Decision']==key] for key in ORDER}
    cleaned=groups[ORDER[0]]+groups[ORDER[1]]
    conservative=cleaned+groups[ORDER[2]]
    prior_count=len(records)

    stages=[
        ('S0','Rows in uploaded provisional selection',len(originals),''),
        ('S1','Deduplicated unique PMIDs',len(records),len(originals)),
        ('S2','Flag likely unrelated prediction tasks; keep uncertain records',len(conservative),len(records)),
        ('S3','Strong and plausible subset (optional priority queue)',len(cleaned),len(conservative)),
    ]
    report=[]
    print('\nSTAGE-BY-STAGE REDUCTIONS')
    for number,description,count,previous in stages:
        removed=previous-count if previous!='' else ''
        pct=round(100*removed/previous,2) if previous not in ('',0) else ''
        report.append({'Stage':number,'Criterion':description,'Remaining':count,'Removed_From_Previous':removed,'Percent_Removed':pct})
        if removed=='':print('  {:<3} {:<70} {:>6,}'.format(number,description,count))
        else:print('  {:<3} {:<70} {:>6,} | -{:>4,} ({:>5.2f}%)'.format(number,description,count,removed,pct))
    print('\nEVIDENCE GROUPS')
    for group in ORDER:
        print('  {:<40} {:>5,}'.format(group,len(groups[group])))
    assert len(scored)==sum(len(g) for g in groups.values())
    assert len(cleaned)+len(groups[ORDER[2]])+len(groups[ORDER[3]])==len(records)

    write_csv(OUTPUT/'00_STAGE_COUNTS.csv',report)
    write_csv(OUTPUT/'03_PRIORITY_STRONG_PLAUSIBLE.csv',cleaned)
    write_csv(OUTPUT/'04_STRONG_EVIDENCE.csv',groups[ORDER[0]])
    write_csv(OUTPUT/'05_PLAUSIBLE_EVIDENCE.csv',groups[ORDER[1]])
    write_csv(OUTPUT/'06_UNCERTAIN_FULL_TEXT_REVIEW.csv',groups[ORDER[2]])
    write_csv(OUTPUT/'02_REMOVED_LIKELY_FALSE_POSITIVES.csv',groups[ORDER[3]])
    write_csv(OUTPUT/'07_ALL_RECORDS_WITH_DECISIONS.csv',scored)
    write_csv(OUTPUT/'01_FINAL_SET_FOR_MANUAL_REVIEW.csv',conservative)

    reason_counts=Counter()
    for row in scored:
        for reason in str(row.get('Flagged_False_Positive_Reasons','')).split(' | '):
            if reason:reason_counts[reason]+=1
    write_csv(OUTPUT/'08_FLAG_REASON_COUNTS.csv',[
        {'Reason':reason,'Flagged_Records':count} for reason,count in reason_counts.most_common()
    ],['Reason','Flagged_Records'])

    checklist_path=OUTPUT/'09_MANUAL_ELIGIBILITY_CHECKLIST.csv'
    checklist_fields=['PMID','Title','PubMed_URL','Second_Pass_Decision','Candidate_Viral_Labels',
                      'Viral_Prediction_Abstract_Evidence','Method_Evidence_Sentence',
                      'Trained_Supervised_Model_YES_NO','At_Least_Two_Viral_Identities_YES_NO',
                      'Virus_Identity_Prediction_Target_YES_NO','Original_Research_YES_NO',
                      'No_Duplicate_YES_NO','Decision_INCLUDE_EXCLUDE_UNCLEAR','Notes']
    if not checklist_path.exists():
        write_csv(checklist_path,[{k:r.get(k,'') for k in checklist_fields} for r in conservative],checklist_fields)
    else:
        print('  Existing human-review checklist preserved:',checklist_path.name)

    (OUTPUT/'10_METHODS_AND_LIMITATIONS.txt').write_text(
        'Second-stage title/abstract screening of the original PubMed provisional set.\n'
        'This script uses general scientific terminology and does not request PubMed data.\n'
        'It flags sequence-based viral taxonomy, clear non-viral prediction outcomes, descriptive\n'
        'research without a demonstrated classifier, and apparent single-virus control tasks.\n'
        'Uncertain records and likely false positives are preserved for human checking.\n'
        'A missing method term in an abstract alone is never proof of ineligibility.\n'
        'A/B designations reflect metadata evidence, not confirmed full-text eligibility.\n'
        'Two named viruses in an abstract are not evidence that both are predicted labels.\n'
        'Full-text verification is required before reporting final included studies.\n',
        encoding='utf-8'
    )

    print('\nFINAL SET FOR MANUAL ELIGIBILITY REVIEW:',OUTPUT/'01_FINAL_SET_FOR_MANUAL_REVIEW.csv')
    print('UNCERTAIN QUEUE:',OUTPUT/'06_UNCERTAIN_FULL_TEXT_REVIEW.csv')
    print('REMOVED LIKELY FALSE POSITIVES:',OUTPUT/'02_REMOVED_LIKELY_FALSE_POSITIVES.csv')
    print('FULL AUDIT:',OUTPUT/'07_ALL_RECORDS_WITH_DECISIONS.csv')
    print('Selection is based on general title-and-abstract criteria only.')


run_viral_identity_false_positive_filter()


SECOND-PASS SUPERVISED VIRAL-IDENTITY ELIGIBILITY FILTER
Input: D:\01-WFIRM\01-Papers\27- Review with Patrick\001- revised round 1\01- codes final\VirusIdentity_Search_2012_2026_10_10\08_SELECTED_PROVISIONAL_STUDIES.csv
Output: D:\01-WFIRM\01-Papers\27- Review with Patrick\001- revised round 1\01- codes final\VirusIdentity_Search_2012_2026_10_10\Viral_Identity_FalsePositive_Screen
Stage 0 — Input records: 476
Stage 1 — Unique identifiers: 476
  Reviewed metadata: 100/476
  Reviewed metadata: 200/476
  Reviewed metadata: 300/476
  Reviewed metadata: 400/476
  Reviewed metadata: 476/476

STAGE-BY-STAGE REDUCTIONS
  S0  Rows in uploaded provisional selection                                    476
  S1  Deduplicated unique PMIDs                                                 476 | -   0 ( 0.00%)
  S2  Flag likely unrelated prediction tasks; keep uncertain records            337 | - 139 (29.20%)
  S3  Strong and plausible subset (optional priority queue)                     287 | -  50 (14